# Mixture Critical Points — Milestone 9

The **critical point** of a mixture is the temperature and pressure at which the coexisting liquid and vapor phases become identical. For a pure fluid it is a single point; for a mixture it depends on composition, and locating it is a genuinely hard problem — the usual phase-split calculations degenerate exactly there. This notebook reproduces the four mixture critical points of the research paper's **Tables 4.1–4.2** with the modernized engine's Heidemann–Khalil solver.

## Setup (optional)

The cell below is **commented out by default**. Uncomment it if you want the latest `vle-thermo` from PyPI instead of the version already in your kernel.

In [1]:
# Optional: pull the latest vle-thermo from PyPI.
# Uncomment if you want the newest released version instead of
# whatever is currently in your kernel.
# %pip install --upgrade vle-thermo

## Context — the Heidemann–Khalil criteria

From [Chapter IV §4.1](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-4-validation.md) the mixture critical point is where the second **and** third variations of the total Helmholtz energy vanish along the same composition direction. Writing the tangent-plane function

$$ F(T, V, \mathbf{n}) = \sum_i n_i \ln n_i + \frac{A^{\mathrm{res}}(T,V,\mathbf{n})}{RT}, $$

the two conditions are that the Hessian $Q_{ij} = \partial^2 F / \partial n_i \partial n_j$ has a zero eigenvalue (eigenvector $\mathbf{s}$), and that the cubic form $C = \sum_{ijk} s_i s_j s_k\, \partial^3 F / \partial n_i \partial n_j \partial n_k$ is zero. The engine builds $Q$ and $C$ from **dual-number automatic differentiation** (no finite differences) and solves both conditions simultaneously with a 2-D Newton on $(T, V)$.

## What this milestone built

`vle._engine.critical_point_py(eos, tcs, pcs, omegas, z, t_init=…)` returns `(Tc [K], Pc [kPa], Vc [m³/kmol])` for a two-parameter cubic EOS with classical mixing. Under the hood it lives in `engine/src/flash/critical.rs`.

## Worked example — Table 4.1 / 4.2, Mixture 1

Mixture 1 is an ethane(C₂) / propane(C₃) / n-pentane(nC₅) system. Peng & Robinson report $T_c = 404.43$ K, $P_c = 5552$ kPa. We use standard critical constants (the thesis does not report the exact values it used, and it neglects $k_{ij}$).

In [2]:
import vle._engine as e

# Standard critical constants (Tc [K], Pc [kPa], omega).
COMPONENTS = {
    'C2':  (305.32, 4872.0, 0.0995),
    'C3':  (369.83, 4248.0, 0.1523),
    'nC4': (425.12, 3796.0, 0.2000),
    'nC5': (469.70, 3370.0, 0.2515),
}

def critical(names, z, t_init):
    tcs = [COMPONENTS[n][0] for n in names]
    pcs = [COMPONENTS[n][1] for n in names]
    om  = [COMPONENTS[n][2] for n in names]
    tc, pc, vc = e.critical_point_py(
        e.CubicEos.PR1976, tcs, pcs, om, z, t_init=t_init)
    return tc, pc, vc

tc, pc, vc = critical(['C2', 'C3', 'nC5'], [0.3414, 0.3421, 0.3165], 405.0)
print(f'Mixture 1:  Tc = {tc:.2f} K   Pc = {pc:.1f} kPa   Vc = {vc:.4f} m3/kmol')
print(f'Reported :  Tc = 404.43 K   Pc = 5552 kPa')

Mixture 1:  Tc = 404.25 K   Pc = 5540.1 kPa   Vc = 0.2218 m3/kmol
Reported :  Tc = 404.43 K   Pc = 5552 kPa


The engine's $T_c$ and $P_c$ land within ~1% of the Peng & Robinson values — the residual difference is exactly what the thesis attributes to the unreported critical constants and the neglected $k_{ij}$.

### All four Table 4.2 mixtures

We reproduce the full table and compare against the reported values, pinning the agreement with assertions so the notebook fails loudly if the engine ever regresses.

In [3]:
cases = [
    # (name, components, z, t_init, Tc_ref, Pc_ref)
    ('Mix 1', ['C2', 'C3', 'nC5'], [0.3414, 0.3421, 0.3165], 405.0, 404.43, 5552.0),
    ('Mix 2', ['C3', 'nC4', 'nC5'], [0.3276, 0.3398, 0.3326], 430.0, 430.72, 4174.0),
    ('Mix 4', ['C2', 'C3', 'nC4', 'nC5'], [0.2542, 0.2547, 0.2554, 0.2357], 410.0, 410.74, 5063.0),
]

print(f"{'case':7} {'Tc calc':>9} {'Tc ref':>8} {'%':>6}   {'Pc calc':>9} {'Pc ref':>8} {'%':>6}")
for name, names, z, t0, tc_ref, pc_ref in cases:
    tc, pc, _ = critical(names, z, t0)
    et = abs(tc - tc_ref) / tc_ref * 100
    ep = abs(pc - pc_ref) / pc_ref * 100
    print(f'{name:7} {tc:9.2f} {tc_ref:8.2f} {et:6.2f}   {pc:9.1f} {pc_ref:8.1f} {ep:6.2f}')
    # Thesis band: the reported errors are all < 1.6% in T and ~5% in P.
    assert et < 2.0, f'{name} Tc error {et:.2f}% too large'
    assert ep < 6.0, f'{name} Pc error {ep:.2f}% too large'
print('\nAll mixtures within the Chapter IV band.')

case      Tc calc   Tc ref      %     Pc calc   Pc ref      %
Mix 1      404.25   404.43   0.04      5540.1   5552.0   0.21
Mix 2      430.60   430.72   0.03      4167.8   4174.0   0.15
Mix 4      410.59   410.74   0.04      5054.4   5063.0   0.17

All mixtures within the Chapter IV band.


> **Note.** Table 4.2's Mixture 3 (a CO₂/H₂S/methane system) has a reported 5% $P_c$ error even in the thesis, driven by the missing $k_{ij}$ for the strongly non-ideal CO₂/H₂S pair. It is omitted from the pinned set above for that reason; try it yourself in Exercise 2.

## Exercise 1 — how does the critical point move with composition?

Trace the critical temperature of the ethane/n-pentane **binary** as the ethane mole fraction goes from 0.1 to 0.9. Plot $T_c$ vs composition. Physically, $T_c$ should interpolate between the two pure critical temperatures (305 K and 470 K), but **not** linearly.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# TODO: for each ethane fraction x in np.linspace(0.1, 0.9, 9),
#   call critical(['C2', 'nC5'], [x, 1 - x], t_init=...) and collect Tc.
#   Then plot Tc vs x.


<details><summary>Solution</summary>

```python
xs = np.linspace(0.1, 0.9, 9)
tcs_mix = []
for x in xs:
    # A mole-fraction-average Tc is a good initial guess.
    t0 = x * 305.32 + (1 - x) * 469.7
    tc, _, _ = critical(['C2', 'nC5'], [x, 1 - x], t0)
    tcs_mix.append(tc)
plt.plot(xs, tcs_mix, 'o-')
plt.xlabel('ethane mole fraction'); plt.ylabel('mixture Tc (K)')
plt.title('Critical temperature of ethane/n-pentane'); plt.grid(True)
plt.show()
```
The curve bows below the straight line between the pure Tc's — a signature of the mixture's non-ideality.
</details>

## Exercise 2 — the effect of the equation of state

Recompute Mixture 1's critical point with the **RKS** EOS (`e.CubicEos.RKS1972`) instead of PR. Do the critical $T$ and $P$ change? Which is closer to the Peng & Robinson reference (and why would you expect PR to be)?

In [5]:
# TODO: call e.critical_point_py with e.CubicEos.RKS1972 for Mixture 1
# and compare Tc/Pc against the PR result and the 404.43 K / 5552 kPa
# reference.


<details><summary>Solution</summary>

```python
names, z = ['C2', 'C3', 'nC5'], [0.3414, 0.3421, 0.3165]
tcs = [COMPONENTS[n][0] for n in names]
pcs = [COMPONENTS[n][1] for n in names]
om  = [COMPONENTS[n][2] for n in names]
for eos in (e.CubicEos.PR1976, e.CubicEos.RKS1972):
    tc, pc, _ = e.critical_point_py(eos, tcs, pcs, om, z, t_init=405.0)
    print(f'{eos}:  Tc = {tc:.2f} K   Pc = {pc:.1f} kPa')
```
Peng & Robinson themselves used PR, so the PR result is the fair comparison; RKS shifts $P_c$ noticeably because its critical compressibility (0.333) differs from PR's (0.307).
</details>

## References

- Research paper [Chapter IV §4.1 — Critical Point Calculations](https://github.com/miguelju/vle/blob/main/docs/en/research-paper/chapter-4-validation.md) (Tables 4.1–4.2).
- (16) Heidemann, R. A.; Khalil, A. M. *The Calculation of Critical Points.* AIChE J. **1980**, 26 (5), 769.
- (15) Peng, D.-Y.; Robinson, D. B. — the reference critical points.
- Algorithm details: [`MODERNIZATION_PLAN.md`](https://github.com/miguelju/vle/blob/main/docs/plans/MODERNIZATION_PLAN.md) §G (Analytical / dual-number Helmholtz derivatives) and `engine/src/flash/critical.rs`.
